In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from tqdm import tqdm
import copy # Necesario para el Early Stopping

# --- 1. CONFIGURACIÓN Y DATOS ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

df = pd.read_csv('../data/HAM10000_metadata_prepared.csv')
label_map = {clase: idx for idx, clase in enumerate(df['dx'].unique())}
df['label'] = df['dx'].map(label_map)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

class SkinCancerDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.dataframe)
    def __getitem__(self, idx):
        img = Image.open(self.dataframe.loc[idx, 'image_path']).convert('RGB')
        label = self.dataframe.loc[idx, 'label']
        if self.transform: img = self.transform(img)
        return img, label

# Data Augmentation (Mantenemos rotaciones y volteos para generalizar bien)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(SkinCancerDataset(train_df, transform=train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(SkinCancerDataset(val_df, transform=val_transform), batch_size=32, shuffle=False)

# --- 2. EL NUEVO MODELO: EfficientNet-B0 ---
print("Descargando pesos de EfficientNet-B0...")
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# A diferencia de ResNet, en EfficientNet la última capa se llama classifier[1]
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 7)
model = model.to(device)

# --- 3. PESOS, OPTIMIZADOR Y SCHEDULER ---
class_weights = compute_class_weight('balanced', classes=np.unique(train_df['label']), y=train_df['label'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Scheduler: Bajará el learning rate un 90% (factor=0.1) si pasamos 3 épocas sin mejorar (patience=3)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3)

# --- 4. BUCLE AVANZADO CON EARLY STOPPING ---
epochs = 20
best_acc = 0.0
patience_early_stop = 5  # Si pasa 5 épocas sin mejorar en absoluto, cortamos
epochs_no_improve = 0

print("\nIniciando entrenamiento (Máx 20 épocas)...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{running_loss/len(train_loader):.4f}"})
        
    # Fase de validación
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    epoch_acc = 100 * correct / total
    print(f"Precisión en Validación: {epoch_acc:.2f}%")
    
    # Comprobar el Scheduler
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(epoch_acc)
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr < current_lr:
        print(f"[Scheduler] Learning Rate reducido a {new_lr}")

    # Lógica del Early Stopping
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        torch.save(model.state_dict(), '../data/best_efficientnet_model.pth')
        epochs_no_improve = 0
        print("¡Nuevo mejor modelo guardado (best_efficientnet_model.pth)!")
    else:
        epochs_no_improve += 1
        print(f"Sin mejora durante {epochs_no_improve} época(s).")
        if epochs_no_improve >= patience_early_stop:
            print(f"Early stopping disparado. El modelo ya no aprende más.")
            break

print(f"\nEntrenamiento finalizado. Mejor precisión histórica: {best_acc:.2f}%")

Entrenando en: cuda
Descargando pesos de EfficientNet-B0...


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\polor/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:01<00:00, 11.6MB/s]



Iniciando entrenamiento (Máx 20 épocas)...


Epoch 1/20 [Train]: 100%|██████████| 251/251 [01:08<00:00,  3.68it/s, loss=1.3248]


Precisión en Validación: 55.12%
¡Nuevo mejor modelo guardado (best_efficientnet_model.pth)!


...

Precisión en Validación: 80.68%
Sin mejora durante 1 época(s).


Epoch 19/20 [Train]: 100%|██████████| 251/251 [01:08<00:00,  3.66it/s, loss=0.2276]


Precisión en Validación: 82.68%
Sin mejora durante 2 época(s).


Epoch 20/20 [Train]: 100%|██████████| 251/251 [01:24<00:00,  2.98it/s, loss=0.2124]


Precisión en Validación: 81.63%
Sin mejora durante 3 época(s).

Entrenamiento finalizado. Mejor precisión histórica: 82.83%
